# Exploratory Data Analysis - IT Support Cost Optimization
This notebook performs exploratory analysis on cleaned IT support ticket data to uncover cost drivers, satisfaction bottlenecks, escalation trends, and engineer performance.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

# Load dataset
df = pd.read_csv('../data/cleaned_tickets.csv')
df['created_at'] = pd.to_datetime(df['created_at'])
df.head()

## 1. Ticket Cost Analysis
We analyze the total and average cost of tickets broken down by issue category, priority, and client type.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Cost by Category
cat_cost = df.groupby('issue_category')['total_cost'].mean().sort_values(ascending=False).reset_index()
sns.barplot(ax=axes[0], data=cat_cost, x='total_cost', y='issue_category', palette='viridis')
axes[0].set_title('Average Cost by Issue Category')
axes[0].set_xlabel('Average Cost ($)')

# Cost by Priority
prio_cost = df.groupby('priority')['total_cost'].mean().reindex(['Low', 'Medium', 'High', 'Critical']).reset_index()
sns.barplot(ax=axes[1], data=prio_cost, x='priority', y='total_cost', palette='magma')
axes[1].set_title('Average Cost by Priority')
axes[1].set_ylabel('Average Cost ($)')

# Cost by Client Type
client_cost = df.groupby('client_type')['total_cost'].mean().sort_values(ascending=False).reset_index()
sns.barplot(ax=axes[2], data=client_cost, x='client_type', y='total_cost', palette='rocket')
axes[2].set_title('Average Cost by Client Type')
axes[2].set_ylabel('Average Cost ($)')

plt.tight_layout()
plt.savefig('../reports/figures/cost_analysis.png', dpi=300)
plt.show()

**Business Takeaway:**
1. **Security & Cloud categories** are the primary drivers of average ticket costs, representing highly complex issues that require specialized skills.
2. **Critical and High priority tickets** cost significantly more than Low or Medium issues, due to longer hours spent and the usage of more expensive Senior engineers.
3. **Enterprise client tickets** have a higher average cost compared to Small or Medium clients, indicating higher support intensity and tighter response standards.

## 2. Escalation Trends & Impact
This section evaluates how ticket escalations affect support costs and customer satisfaction.

In [ ]:
df['is_escalated'] = (df['escalation_count'] > 0).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Escalation vs Cost
sns.boxplot(ax=axes[0], data=df, x='is_escalated', y='total_cost', palette='Set2')
axes[0].set_title('Impact of Escalation on Total Cost')
axes[0].set_xticklabels(['Not Escalated', 'Escalated'])
axes[0].set_ylabel('Total Cost ($)')

# Escalation vs Satisfaction
sat_esc = df.groupby('escalation_count')['customer_satisfaction_score'].mean().reset_index()
sns.lineplot(ax=axes[1], data=sat_esc, x='escalation_count', y='customer_satisfaction_score', marker='o', color='red', linewidth=2.5)
axes[1].set_title('Satisfaction Score by Escalation Count')
axes[1].set_xlabel('Number of Escalations')
axes[1].set_ylabel('Average CSAT (1-5)')
axes[1].set_ylim(1, 5)

plt.tight_layout()
plt.savefig('../reports/figures/escalation_impact.png', dpi=300)
plt.show()

**Business Takeaway:**
1. **Cost Premium:** Escalated tickets cost significantly more on average due to multiple engineers being involved and more hours accumulated.
2. **Satisfaction Drop:** Customer satisfaction declines sharply with each additional escalation. Tickets with 0 escalations average near ~4.4 CSAT, whereas tickets with 2 or more escalations drop below 3.0 CSAT, suggesting customer frustration grows exponentially with delays.

## 3. Customer Satisfaction Distribution & Drivers
We look at how satisfaction is distributed and what major variables influence it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Satisfaction distribution
sns.countplot(ax=axes[0], data=df, x='customer_satisfaction_score', palette='RdYlGn')
axes[0].set_title('Distribution of Customer Satisfaction Scores')
axes[0].set_xlabel('CSAT Score (1-5)')

# Satisfaction by First Contact Resolution
sns.barplot(ax=axes[1], data=df, x='resolved_first_contact', y='customer_satisfaction_score', palette='coolwarm')
axes[1].set_title('CSAT by First Contact Resolution Status')
axes[1].set_xticklabels(['No', 'Yes'])
axes[1].set_ylabel('Average CSAT')
axes[1].set_ylim(0, 5)

plt.tight_layout()
plt.savefig('../reports/figures/satisfaction_distribution.png', dpi=300)
plt.show()

**Business Takeaway:**
1. **Bimodal CSAT:** While the majority of customers rate support at a 4 or 5, there is a notable tail of 1, 2, and 3 scores representing failed SLA outcomes or multiple escalations.
2. **FCR Value:** Resolving issues on first contact (`resolved_first_contact = True`) yields a massive satisfaction boost, keeping average CSAT near ~4.6/5.0, compared to ~3.2/5.0 when follow-ups are needed.

## 4. Engineer Performance by Experience Level
We compare the costs, resolution speeds, and customer satisfaction results of Junior, Mid, and Senior engineers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Cost/Hour by Experience
sns.barplot(ax=axes[0], data=df, x='engineer_experience_level', y='cost_per_engineer_hour', order=['Junior', 'Mid', 'Senior'], palette='pastel')
axes[0].set_title('Average Cost per Hour')
axes[0].set_ylabel('Cost/Hour ($)')

# Resolution Time by Experience
sns.barplot(ax=axes[1], data=df, x='engineer_experience_level', y='resolution_time_minutes', order=['Junior', 'Mid', 'Senior'], palette='pastel')
axes[1].set_title('Average Resolution Time (Mins)')
axes[1].set_ylabel('Resolution Time (Minutes)')

# Satisfaction by Experience
sns.barplot(ax=axes[2], data=df, x='engineer_experience_level', y='customer_satisfaction_score', order=['Junior', 'Mid', 'Senior'], palette='pastel')
axes[2].set_title('Average Customer Satisfaction (CSAT)')
axes[2].set_ylabel('CSAT Score')
axes[2].set_ylim(0, 5)

plt.tight_layout()
plt.savefig('../reports/figures/engineer_performance.png', dpi=300)
plt.show()

**Business Takeaway:**
1. **Experience Premium:** Senior engineers have the highest billing rates (~$110/hr) but resolve issues in almost half the time taken by Junior engineers.
2. **CSAT Gain:** Senior engineers achieve significantly higher average CSAT scores (~4.6) compared to Junior engineers (~3.8). This demonstrates the value of routing complex, high-urgency tickets directly to senior staff rather than letting them bounce from junior levels.

## 5. Ticket Volume and Cost Trends
We trace ticket volume and total costs over time to look for weekly or seasonal trends.

In [ ]:
df['month'] = df['created_at'].dt.to_period('M')
monthly_stats = df.groupby('month').agg(ticket_count=('ticket_id', 'count'), total_monthly_cost=('total_cost', 'sum')).reset_index()
monthly_stats['month'] = monthly_stats['month'].astype(str)

fig, ax1 = plt.subplots(figsize=(12, 5))

color = 'tab:blue'
ax1.set_xlabel('Month (2025)')
ax1.set_ylabel('Ticket Volume', color=color)
sns.barplot(data=monthly_stats, x='month', y='ticket_count', ax=ax1, color=color, alpha=0.6)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticklabels(monthly_stats['month'], rotation=45)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Total Cost ($)', color=color)
sns.lineplot(data=monthly_stats, x='month', y='total_monthly_cost', ax=ax2, color=color, marker='o', linewidth=2.5)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Monthly Ticket Volume and Total Cost Trend')
plt.tight_layout()
plt.savefig('../reports/figures/volume_cost_trends.png', dpi=300)
plt.show()

**Business Takeaway:**
1. **Volume Stability:** Ticket volume remains relatively stable month-to-month, meaning resource planning can be modeled around consistent weekly averages rather than extreme seasonal spikes.
2. **Cost Correlation:** Total support cost strictly tracks ticket volume, demonstrating that overall operational efficiency is highly dependent on reducing individual ticket costs.

## 6. Correlation Heatmap
A correlation matrix helps identify which variables move together.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=300)
plt.show()

**Business Takeaway:**
1. **Total Cost Key Drivers:** Total cost is extremely highly correlated with `engineer_hours_spent`, which is the strongest lever for cost optimization.
2. **Satisfaction Drivers:** CSAT is negatively correlated with `escalation_count` and `resolution_time_minutes`, proving that ticket delays and handoffs damage customer satisfaction.